# 🏥 Medical Document Risk Analyzer
> Detects forged / tampered medical certificates using MobileNetV2 Transfer Learning + OCR.

**GitHub:** [Aryan1826/medical-document-risk-analyzer](https://github.com/Aryan1826/medical-document-risk-analyzer)

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q pytesseract pyngrok
!apt-get install -y tesseract-ocr -q

import sys, tensorflow as tf
print(f'Python     : {sys.version}')
print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')
print('Setup complete!')

---
## Step 2: Imports

In [ ]:
import os, io, re, sys, random, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Input,
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger,
)
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('All imports OK!')

---
## Step 3: Configuration
> **Only change `DATASET_PATH`** — everything else is set automatically.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  USER SETTINGS  ← only edit this block
# ─────────────────────────────────────────────────────────────
DATASET_PATH = 'dataset'   # folder must contain real/ and fake/
# Example (Google Drive): '/content/drive/MyDrive/my_dataset'
# ─────────────────────────────────────────────────────────────

IMG_SIZE      = (224, 224)
BATCH_SIZE    = 32
PHASE1_EPOCHS = 10   # frozen base  – fast head training
PHASE2_EPOCHS = 10   # fine-tune top 50 layers
LR            = 1e-3
FINE_TUNE_LR  = 1e-5

DATASET_DIR = Path(DATASET_PATH)
REAL_DIR    = DATASET_DIR / 'real'
FAKE_DIR    = DATASET_DIR / 'fake'
MODEL_DIR   = Path('models')
MODEL_PATH  = MODEL_DIR / 'best_model.keras'
LOG_PATH    = MODEL_DIR / 'training_log.csv'
THRESH_PATH = MODEL_DIR / 'optimal_threshold.txt'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset : {DATASET_DIR.resolve()}')
print(f'Model   : {MODEL_PATH}')
print(f'IMG     : {IMG_SIZE}  |  Batch: {BATCH_SIZE}')
print(f'Epochs  : {PHASE1_EPOCHS} (frozen) + {PHASE2_EPOCHS} (fine-tune)')

---
## Step 4: Data Validation
Checks your dataset has the right structure and enough images before training.

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

def count_images(folder):
    if not folder.exists(): return 0, []
    imgs = [p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS]
    return len(imgs), imgs

n_real, real_paths = count_images(REAL_DIR)
n_fake, fake_paths = count_images(FAKE_DIR)
total = n_real + n_fake

print('=' * 50)
print('  DATASET VALIDATION')
print('=' * 50)
print(f'  REAL  : {n_real} images')
print(f'  FAKE  : {n_fake} images')
print(f'  Total : {total}')
print('=' * 50)

errors = []
if not REAL_DIR.exists(): errors.append(f'Missing: {REAL_DIR}')
if not FAKE_DIR.exists(): errors.append(f'Missing: {FAKE_DIR}')
if n_real < 10:           errors.append(f'Too few REAL images ({n_real}). Need ≥ 10.')
if n_fake < 10:           errors.append(f'Too few FAKE images ({n_fake}). Need ≥ 10.')
if errors:
    for e in errors: print(f'  [ERROR] {e}')
    raise RuntimeError('Fix the errors above and re-run.')

ratio = min(n_real, n_fake) / max(n_real, n_fake)
if ratio < 0.70:
    print(f'  [WARN] Imbalance ratio={ratio:.2f}. Class weights will compensate.')
else:
    print(f'  [OK]   Balance ratio: {ratio:.2f}')
print('\nValidation passed — ready to train!')

---
## Step 5: Visualise Samples

In [ ]:
real_imgs = sorted([p for p in REAL_DIR.iterdir() if p.suffix.lower() in IMG_EXTS])[:4]
fake_imgs = sorted([p for p in FAKE_DIR.iterdir() if p.suffix.lower() in IMG_EXTS])[:4]
n_cols = max(len(real_imgs), len(fake_imgs))

if n_cols == 0:
    print('No images to display.')
else:
    fig, axes = plt.subplots(2, n_cols, figsize=(5*n_cols, 8))
    if n_cols == 1: axes = [[axes[0]], [axes[1]]]
    fig.suptitle('REAL (top)  vs  FAKE (bottom)', fontsize=14)
    for col in range(n_cols):
        for row, (paths, color) in enumerate([(real_imgs,'green'),(fake_imgs,'red')]):
            ax = axes[row][col]
            if col < len(paths):
                ax.imshow(Image.open(paths[col]))
                ax.set_title(paths[col].name, fontsize=8, color=color)
            ax.axis('off')
    plt.tight_layout(); plt.show()

---
## Step 6: Data Pipeline
ImageDataGenerator with strong augmentation for training, rescale-only for validation.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255, validation_split=0.20,
    rotation_range=20, zoom_range=0.20, shear_range=0.20,
    width_shift_range=0.10, height_shift_range=0.10,
    horizontal_flip=True, brightness_range=[0.80, 1.20],
    fill_mode='nearest',
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.20)

train_gen = train_datagen.flow_from_directory(
    str(DATASET_DIR), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='training', shuffle=True, seed=SEED,
)
val_gen = val_datagen.flow_from_directory(
    str(DATASET_DIR), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation', shuffle=False, seed=SEED,
)

CLASS_INDICES = train_gen.class_indices
# Keras assigns labels alphabetically: fake=0, real=1
# Model sigmoid = P(real)  →  P(fake) = 1 − output
print(f'Class indices  : {CLASS_INDICES}')
print(f'Train images   : {train_gen.n}  |  Val images: {val_gen.n}')

x, y = next(iter(train_gen))
print(f'Batch shape    : {x.shape}  |  pixel range [{x.min():.2f}, {x.max():.2f}]')

real_idx = CLASS_INDICES.get('real', 1)
fake_idx = CLASS_INDICES.get('fake', 0)
CLASS_WEIGHTS = {
    real_idx: total / (2 * max(n_real, 1)),
    fake_idx: total / (2 * max(n_fake, 1)),
}
print(f'Class weights  : {CLASS_WEIGHTS}')

---
## Step 7: Model — MobileNetV2 Transfer Learning

In [ ]:
def build_model(freeze_base=True):
    base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = not freeze_base
    if not freeze_base:
        for l in base.layers[:-50]: l.trainable = False
        for l in base.layers[-50:]: l.trainable = True

    inp = Input(shape=(*IMG_SIZE, 3), name='input')
    x   = base(inp, training=not freeze_base)
    x   = GlobalAveragePooling2D()(x)
    x   = Dense(256, activation='relu')(x)
    x   = BatchNormalization()(x)
    x   = Dropout(0.50)(x)
    x   = Dense(128, activation='relu')(x)
    x   = Dropout(0.40)(x)
    out = Dense(1, activation='sigmoid', name='output')(x)
    return Model(inputs=inp, outputs=out, name='MedicalDocRiskAnalyzer'), base

model, base_model = build_model(freeze_base=True)
total_p     = model.count_params()
trainable_p = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'Total params     : {total_p:,}')
print(f'Trainable (head) : {trainable_p:,}')
print(f'Frozen (base)    : {total_p - trainable_p:,}')
model.summary(line_length=80)

---
## Step 8: Training
Phase 1 trains only the head (base frozen). Phase 2 fine-tunes the top 50 MobileNetV2 layers.

In [ ]:
import collections
loss_fn = tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1)
metrics = ['accuracy', tf.keras.metrics.AUC(name='auc'),
           tf.keras.metrics.Precision(name='precision'),
           tf.keras.metrics.Recall(name='recall')]

# ── Phase 1: train head only ──────────────────────────────────
print('Phase 1 — training head (base frozen)...')
model.compile(optimizer=Adam(LR), loss=loss_fn, metrics=metrics)

h1 = model.fit(
    train_gen, epochs=PHASE1_EPOCHS, validation_data=val_gen,
    class_weight=CLASS_WEIGHTS, verbose=1,
    callbacks=[
        ModelCheckpoint(str(MODEL_PATH), monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.4, patience=3, min_lr=1e-8, verbose=1),
        CSVLogger(str(LOG_PATH), append=False),
    ]
)
print(f'Phase 1 best val_accuracy = {max(h1.history["val_accuracy"]):.4f}')

# ── Phase 2: fine-tune top 50 layers ─────────────────────────
print('\nPhase 2 — fine-tuning top 50 MobileNetV2 layers...')
for l in base_model.layers[:-50]: l.trainable = False
for l in base_model.layers[-50:]: l.trainable = True
model.compile(optimizer=Adam(FINE_TUNE_LR), loss=loss_fn, metrics=metrics)

h2 = model.fit(
    train_gen, initial_epoch=PHASE1_EPOCHS,
    epochs=PHASE1_EPOCHS+PHASE2_EPOCHS, validation_data=val_gen,
    class_weight=CLASS_WEIGHTS, verbose=1,
    callbacks=[
        ModelCheckpoint(str(MODEL_PATH), monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-9, verbose=1),
        CSVLogger(str(LOG_PATH), append=True),
    ]
)
print(f'Phase 2 best val_accuracy = {max(h2.history["val_accuracy"]):.4f}')

# Merge histories
history = collections.defaultdict(list)
for k,v in h1.history.items(): history[k].extend(v)
for k,v in h2.history.items(): history[k].extend(v)
history = dict(history)
print('Training complete!')

---
## Step 9: Training History

In [ ]:
ep = range(1, len(history['accuracy'])+1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Medical Document Risk Analyzer — Training History', fontsize=13)

for ax in axes:
    ax.axvline(PHASE1_EPOCHS+0.5, color='gray', ls='--', lw=1, label='Phase boundary')

axes[0].plot(ep, history['accuracy'], 'b-o', ms=4, label='Train')
axes[0].plot(ep, history['val_accuracy'], 'r-o', ms=4, label='Val')
axes[0].set_title('Accuracy'); axes[0].set_ylim(0,1.05); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['loss'], 'b-o', ms=4, label='Train')
axes[1].plot(ep, history['val_loss'], 'r-o', ms=4, label='Val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)

if 'auc' in history:
    axes[2].plot(ep, history['auc'], 'b-o', ms=4, label='Train AUC')
    axes[2].plot(ep, history['val_auc'], 'r-o', ms=4, label='Val AUC')
    axes[2].set_title('AUC'); axes[2].set_ylim(0,1.05); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('models/training_history.png', dpi=120, bbox_inches='tight')
plt.show()
best = int(np.argmax(history['val_accuracy']))
print(f'Best epoch       : {best+1}')
print(f'Best val_accuracy: {history["val_accuracy"][best]:.4f}')
if 'val_auc' in history:
    print(f'Best val_AUC     : {history["val_auc"][best]:.4f}')

---
## Step 10: Evaluation & Optimal Threshold
Youden's J statistic finds the threshold that best balances sensitivity and specificity.

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix, classification_report, roc_auc_score
import itertools

best_model = load_model(str(MODEL_PATH))
val_gen.reset()
y_pred_raw = best_model.predict(val_gen, verbose=0).flatten()
y_true     = val_gen.classes[:len(y_pred_raw)]
fake_idx_  = CLASS_INDICES.get('fake', 0)
y_score    = 1 - y_pred_raw   # P(fake)

# Optimal threshold via Youden's J
fpr, tpr, thresholds = roc_curve(y_true == fake_idx_, y_score)
j = tpr - fpr
OPTIMAL_THRESHOLD = float(thresholds[np.argmax(j)])
THRESH_PATH.write_text(str(OPTIMAL_THRESHOLD))
print(f'Optimal threshold (Youden J): {OPTIMAL_THRESHOLD:.4f}')
print(f'AUC: {roc_auc_score(y_true == fake_idx_, y_score):.4f}')

y_pred = (y_score >= OPTIMAL_THRESHOLD).astype(int)
print()
print(classification_report(y_true == fake_idx_, y_pred, target_names=['REAL','FAKE']))

# Confusion matrix
cm = confusion_matrix(y_true == fake_idx_, y_pred)
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['REAL','FAKE']); ax.set_yticklabels(['REAL','FAKE'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix  (thr={OPTIMAL_THRESHOLD:.3f})')
for i,j_ in itertools.product(range(2),range(2)):
    ax.text(j_,i,cm[i,j_],ha='center',va='center',color='white' if cm[i,j_]>cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

---
## Step 11: Predict a Single Image

In [ ]:
def _load_threshold():
    try: return float(THRESH_PATH.read_text().strip())
    except: return 0.50

def predict_image(image_input, threshold=None, verbose=True):
    """
    Args:
        image_input: file path (str/Path), PIL Image, or numpy array
    Returns:
        dict with keys: label, confidence, fake_prob, real_prob, decision
    """
    thr = threshold if threshold is not None else _load_threshold()

    if isinstance(image_input, (str, Path)):
        img = Image.open(str(image_input)).convert('RGB').resize(IMG_SIZE)
    elif isinstance(image_input, Image.Image):
        img = image_input.convert('RGB').resize(IMG_SIZE)
    else:
        img = Image.fromarray(np.uint8(image_input)).convert('RGB').resize(IMG_SIZE)

    arr   = np.array(img, dtype=np.float32) / 255.0
    batch = np.expand_dims(arr, 0)

    raw_out   = float(best_model.predict(batch, verbose=0)[0][0])
    real_prob = raw_out
    fake_prob = 1.0 - raw_out
    label     = 'FAKE' if fake_prob >= thr else 'REAL'
    confidence= fake_prob if label == 'FAKE' else real_prob
    decision  = f'Prediction: {label} (confidence: {confidence:.2f})'

    if verbose:
        print(decision)
        print(f'  P(FAKE) = {fake_prob:.4f}   P(REAL) = {real_prob:.4f}   threshold = {thr:.4f}')

    return {'label': label, 'confidence': confidence,
            'fake_prob': fake_prob, 'real_prob': real_prob, 'decision': decision}
print('predict_image() ready.')

---
## Step 12: Upload & Test

In [ ]:
from google.colab import files as colab_files
from IPython.display import display

print('Upload a medical certificate image to test:')
uploaded = colab_files.upload()

for fname, data in uploaded.items():
    img = Image.open(io.BytesIO(data)).convert('RGB')
    result = predict_image(img, verbose=True)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img); axes[0].axis('off')
    axes[0].set_title(result['decision'],
                      color='red' if result['label']=='FAKE' else 'green', fontsize=12)

    bar_data = [result['real_prob'], result['fake_prob']]
    bars = axes[1].barh(['REAL','FAKE'], bar_data, color=['green','red'])
    axes[1].set_xlim(0,1); axes[1].set_xlabel('Probability')
    axes[1].set_title('Risk Breakdown'); axes[1].grid(axis='x', alpha=0.3)
    for bar, val in zip(bars, bar_data):
        axes[1].text(val+0.01, bar.get_y()+bar.get_height()/2,
                     f'{val:.2%}', va='center', fontsize=10)

    plt.tight_layout(); plt.show()